In [38]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from rich.console import Console
from rich.table import Table

In [39]:
imdb_charts = {
    "Top Movies": "https://www.imdb.com/chart/top/",
    "Most Popular Movies": "https://www.imdb.com/chart/moviemeter/",
    "Box Office": "https://www.imdb.com/chart/boxoffice/",
    "Top TV Shows": "https://www.imdb.com/chart/toptv/",
    "Most Popular TV Shows": "https://www.imdb.com/chart/tvmeter/",
}

In [40]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.5",
}

console = Console()
scraped_data = []

In [41]:
def fetch_html(url, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            if response.status_code == 200:
                return response.text
            console.print(f"[bold red]Failed to retrieve data (Attempt {attempt+1}/{max_retries}): {response.status_code}[/bold red]")
        except requests.exceptions.RequestException as e:
            console.print(f"[bold red]Request failed (Attempt {attempt+1}/{max_retries}): {e}[/bold red]")
    return None

In [42]:
def scrape_imdb_chart(chart_name, url):
    html = fetch_html(url)
    if not html:
        console.print(
            f"[bold red]Failed to retrieve {chart_name} after multiple attempts.[/bold red]"
        )
        return

    soup = BeautifulSoup(html, "html.parser")
    movies = soup.find_all("li", class_="ipc-metadata-list-summary-item")

    if not movies:
        console.print(f"[bold red]No movies found for {chart_name}. IMDb might have changed its structure.[/bold red]")
        return

    table = Table(title=f"🎬 {chart_name}", header_style="bold magenta")
    table.add_column("Rank", justify="center", style="cyan")
    table.add_column("Title", justify="left", style="yellow")
    table.add_column("Year", justify="center", style="green")
    table.add_column("IMDb Rating", justify="center", style="blue")

    for i, movie in enumerate(movies[:20]):
        title = movie.find("h3", class_="ipc-title__text")
        year = movie.find("span", class_="cli-title-metadata-item")
        rating = movie.find("span", class_="ipc-rating-star--rating")

        title_text = title.text if title else "N/A"
        year_text = year.text if year else "N/A"
        rating_text = rating.text if rating else "N/A"

        table.add_row(f"[bold]{i+1}[/bold]", title_text, year_text, rating_text)
        scraped_data.append(
            {
                "Chart": chart_name,
                "Rank": i + 1,
                "Title": title_text,
                "Year": year_text,
                "IMDb Rating": rating_text,
            }
        )

    console.print(table)

In [43]:
scrape_imdb_chart("Top Movies", imdb_charts["Top Movies"])

                                   🎬 Top Movies                                    
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┓
┃ Rank ┃ Title                                                ┃ Year ┃ IMDb Rating ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━┩
│  1   │ 1. The Shawshank Redemption                          │ 1994 │     9.3     │
│  2   │ 2. The Godfather                                     │ 1972 │     9.2     │
│  3   │ 3. The Dark Knight                                   │ 2008 │     9.0     │
│  4   │ 4. The Godfather Part II                             │ 1974 │     9.0     │
│  5   │ 5. 12 Angry Men                                      │ 1957 │     9.0     │
│  6   │ 6. The Lord of the Rings: The Return of the King     │ 2003 │     9.0     │
│  7   │ 7. Schindler's List                                  │ 1993 │     9.0     │
│  8   │ 8. Pulp Fiction                                      │ 1994 │     8.9     │
│  9   │ 9. The Lord of the Rings: The Fellowship of the Ring │ 2001 │     8.9     │
│  10  │ 10. The Good, the Bad and the Ugly                   │ 1966 │     8.8     │
│  11  │ 11. Forrest Gump                                     │ 1994 │     8.8     │
│  12  │ 12. The Lord of the Rings: The Two Towers            │ 2002 │     8.8     │
│  13  │ 13. Fight Club                                       │ 1999 │     8.8     │
│  14  │ 14. Inception                                        │ 2010 │     8.8     │
│  15  │ 15. Star Wars: Episode V - The Empire Strikes Back   │ 1980 │     8.7     │
│  16  │ 16. The Matrix                                       │ 1999 │     8.7     │
│  17  │ 17. Goodfellas                                       │ 1990 │     8.7     │
│  18  │ 18. One Flew Over the Cuckoo's Nest                  │ 1975 │     8.7     │
│  19  │ 19. Interstellar                                     │ 2014 │     8.7     │
│  20  │ 20. Se7en                                            │ 1995 │     8.6     │
└──────┴──────────────────────────────────────────────────────┴──────┴─────────────┘

In [44]:
scrape_imdb_chart("Most Popular Movies", imdb_charts["Most Popular Movies"])

                     🎬 Most Popular Movies                     
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┓
┃ Rank ┃ Title                            ┃ Year ┃ IMDb Rating ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━┩
│  1   │ Captain America: Brave New World │ 2025 │     6.1     │
│  2   │ The Gorge                        │ 2025 │     6.8     │
│  3   │ Thunderbolts*                    │ 2025 │     N/A     │
│  4   │ Nosferatu                        │ 2024 │     7.3     │
│  5   │ Bridget Jones: Mad About the Boy │ 2025 │     6.8     │
│  6   │ Anora                            │ 2024 │     7.7     │
│  7   │ The Brutalist                    │ 2024 │     7.9     │
│  8   │ Companion                        │ 2025 │     7.2     │
│  9   │ How to Train Your Dragon         │ 2025 │     N/A     │
│  10  │ The Substance                    │ 2024 │     7.3     │
│  11  │ Kinda Pregnant                   │ 2025 │     4.9     │
│  12  │ Jurassic World Rebirth           │ 2025 │     N/A     │
│  13  │ Heart Eyes                       │ 2025 │     6.6     │
│  14  │ The Order                        │ 2024 │     6.8     │
│  15  │ The Fantastic Four: First Steps  │ 2025 │     N/A     │
│  16  │ I'm Still Here                   │ 2024 │     8.8     │
│  17  │ You're Cordially Invited         │ 2025 │     5.5     │
│  18  │ Babygirl                         │ 2024 │     6.0     │
│  19  │ Ne Zha 2                         │ 2025 │     8.3     │
│  20  │ Conclave                         │ 2024 │     7.4     │
└──────┴──────────────────────────────────┴──────┴─────────────┘

In [45]:
scrape_imdb_chart("Box Office", imdb_charts["Box Office"])

                           🎬 Box Office                           
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┓
┃ Rank ┃ Title                               ┃ Year ┃ IMDb Rating ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━┩
│  1   │ 1. Captain America: Brave New World │ N/A  │     6.1     │
│  2   │ 2. Paddington in Peru               │ N/A  │     6.8     │
│  3   │ 3. Heart Eyes                       │ N/A  │     6.6     │
│  4   │ 4. Dog Man                          │ N/A  │     6.4     │
│  5   │ 5. Ne Zha 2                         │ N/A  │     8.3     │
│  6   │ 6. Mufasa: The Lion King            │ N/A  │     6.7     │
│  7   │ 7. Love Hurts                       │ N/A  │     5.4     │
│  8   │ 8. One of Them Days                 │ N/A  │     6.8     │
│  9   │ 9. Companion                        │ N/A  │     7.2     │
│  10  │ 10. Chhaava                         │ N/A  │     8.2     │
└──────┴─────────────────────────────────────┴──────┴─────────────┘

In [46]:
scrape_imdb_chart("Top TV Shows", imdb_charts["Top TV Shows"])

                             🎬 Top TV Shows                             
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Rank ┃ Title                                ┃   Year    ┃ IMDb Rating ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│  1   │ 1. Breaking Bad                      │ 2008–2013 │     9.5     │
│  2   │ 2. Planet Earth II                   │   2016    │     9.4     │
│  3   │ 3. Planet Earth                      │   2006    │     9.4     │
│  4   │ 4. Band of Brothers                  │   2001    │     9.4     │
│  5   │ 5. Chernobyl                         │   2019    │     9.3     │
│  6   │ 6. The Wire                          │ 2002–2008 │     9.3     │
│  7   │ 7. Avatar: The Last Airbender        │ 2005–2008 │     9.3     │
│  8   │ 8. Blue Planet II                    │   2017    │     9.3     │
│  9   │ 9. The Sopranos                      │ 1999–2007 │     9.2     │
│  10  │ 10. Cosmos: A Spacetime Odyssey      │   2014    │     9.2     │
│  11  │ 11. Cosmos                           │   1980    │     9.3     │
│  12  │ 12. Our Planet                       │ 2019–2023 │     9.2     │
│  13  │ 13. Game of Thrones                  │ 2011–2019 │     9.2     │
│  14  │ 14. Bluey                            │   2018–   │     9.3     │
│  15  │ 15. The World at War                 │ 1973–1974 │     9.2     │
│  16  │ 16. Fullmetal Alchemist: Brotherhood │ 2009–2010 │     9.1     │
│  17  │ 17. Life                             │   2009    │     9.1     │
│  18  │ 18. Rick and Morty                   │   2013–   │     9.1     │
│  19  │ 19. The Last Dance                   │   2020    │     9.0     │
│  20  │ 20. The Twilight Zone                │ 1959–1964 │     9.0     │
└──────┴──────────────────────────────────────┴───────────┴─────────────┘

In [47]:
scrape_imdb_chart("Most Popular TV Shows", imdb_charts["Most Popular TV Shows"])

                🎬 Most Popular TV Shows                 
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Rank ┃ Title                ┃   Year    ┃ IMDb Rating ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│  1   │ Severance            │   2022–   │     8.7     │
│  2   │ Paradise             │   2025–   │     7.9     │
│  3   │ Apple Cider Vinegar  │   2025    │     7.3     │
│  4   │ Invincible           │   2021–   │     8.6     │
│  5   │ The White Lotus      │   2021–   │     8.0     │
│  6   │ Cobra Kai            │ 2018–2025 │     8.4     │
│  7   │ The Night Agent      │   2023–   │     7.5     │
│  8   │ Yellowstone          │ 2018–2024 │     8.6     │
│  9   │ High Potential       │   2024–   │     7.6     │
│  10  │ Dexter: Original Sin │   2024–   │     8.3     │
│  11  │ The Recruit          │   2022–   │     7.4     │
│  12  │ Game of Thrones      │ 2011–2019 │     9.2     │
│  13  │ Yellowjackets        │   2021–   │     7.8     │
│  14  │ The Åre Murders      │   2025–   │     6.7     │
│  15  │ Dexter               │ 2006–2013 │     8.6     │
│  16  │ The Rookie           │   2018–   │     8.0     │
│  17  │ Solo Leveling        │   2024–   │     8.4     │
│  18  │ The Pitt             │   2025–   │     8.4     │
│  19  │ Squid Game           │ 2021–2025 │     8.0     │
│  20  │ American Primeval    │   2025    │     8.1     │
└──────┴──────────────────────┴───────────┴─────────────┘